# Preference Tuning: DPO, KTO & GRPO — Week 5

## Learning Objectives
By the end of this notebook you will be able to:
1. Explain why SFT alone is insufficient and what preference tuning adds
2. Distinguish DPO, KTO, and GRPO by data requirements and use cases
3. Build a preference dataset using `PrefRunner.build_dpo_dataset_from_llm`
4. Run a DPO training smoke test on your SFT adapter
5. Explain how GRPO enables training with a reward *function* instead of a reward *model*

## Time Estimate
~30 minutes (10 min reading + 10 min dataset gen + 10 min training)

In [1]:
import sys, importlib, json, os
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv(override=True)

%matplotlib inline

import src.preference_tuner as pref_mod
import src.lora_setup as lora_setup
importlib.reload(pref_mod)
importlib.reload(lora_setup)

from src.preference_tuner import PrefRunner, make_gsm8k_reward_fn
from src.config import FINETUNE_BACKEND, BASE_MODEL_HF
from src.cost_tracker import CostTracker
from src.utils import append_to_reflection

tracker = CostTracker()
print(f"Backend : {FINETUNE_BACKEND}")
print(f"Model   : {BASE_MODEL_HF}")

Backend : mlx
Model   : Qwen/Qwen2.5-0.5B-Instruct


---
## Part 1: Beyond SFT — The Alignment Tax

SFT teaches the model to **imitate** — to produce outputs that look like your training data. But imitation has limits:

- If your training data contains mediocre answers, the model learns to be mediocre
- The model can't learn *why* one answer is better than another
- Sycophancy: the model learns to sound confident and agreeable even when wrong

**Preference tuning** teaches the model to be *preferred*. We show it pairs of responses and signal which is better. The model learns the latent quality signal — not just surface-level style.

### The Three Main Methods

| Method | Data needed | When to use | TRL class |
|--------|-------------|-------------|-----------||
| **DPO** | `(prompt, chosen, rejected)` | Chat quality, helpfulness, reducing sycophancy | `DPOTrainer` |
| **KTO** | `(prompt, completion, label: bool)` | Binary feedback (thumbs up/down), cheapest to collect | `KTOTrainer` |
| **GRPO** | `(prompt)` + reward function | Math, code, verifiable tasks — no reward model needed | `GRPOTrainer` |

### Key intuition

**DPO** (Direct Preference Optimization, Rafailov et al. 2023): instead of training a separate reward model and then doing PPO (the old RLHF way), DPO shows that you can directly optimize the language model policy against preference pairs. It's simpler, more stable, and requires far less compute than PPO-based RLHF.

**KTO** (Kahneman-Tversky Optimization): based on prospect theory — humans feel losses more acutely than gains. KTO only needs a binary label per completion (good/bad), not explicit pairings. Useful when you have click-through data, thumbs up/down ratings, or flagged outputs.

**GRPO** (Group Relative Policy Optimization): generates multiple completions per prompt, scores them with a reward function, and uses the relative scores within the group as the training signal. No reward model — the reward function can be as simple as a regex check.

---
## Part 2: Building a Preference Dataset

We'll use `PrefRunner.build_dpo_dataset_from_llm` to generate preference pairs. For each question:
- **Chosen**: a detailed, specific, well-explained answer (generated with a detailed-answer prompt)
- **Rejected**: a vague, one-sentence answer that lacks specifics

This creates an artificial but principled preference signal: the model learns that thoroughness and specificity are preferred over vagueness.

In [2]:
# Set up the LLM client for generating preference pairs
# We'll use the Claude API (or Ollama fallback)
from src.llm_client import LLMClient

llm_client = LLMClient()  # uses ANTHROPIC_API_KEY from .env
print(f"LLM client ready: {llm_client.default_model}")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
LLM client ready: claude-sonnet-4-6


In [3]:
# Resume domain questions for preference pair generation
RESUME_QUESTIONS = [
    "What is Scott's background in machine learning?",
    "What programming languages and frameworks does Scott use?",
    "Describe Scott's most significant technical project.",
    "What makes Scott a strong candidate for an ML Engineering role?",
    "How does Scott approach learning new technologies?",
]

pref_runner = PrefRunner(method='dpo', base_model_path='../outputs/sft_adapter')

print("Generating 5 DPO preference pairs (this calls the Claude API)...")
dpo_pairs = pref_runner.build_dpo_dataset_from_llm(
    questions=RESUME_QUESTIONS,
    llm_client=llm_client,
)

# Track usage
# Note: LLMClient returns usage stats; we add them manually here
print(f"\nGenerated {len(dpo_pairs)} preference pairs")

[preference_tuner] Initialized PrefRunner method=dpo model=../outputs/sft_adapter
Generating 5 DPO preference pairs (this calls the Claude API)...
[preference_tuner] Building DPO dataset from 5 questions...
[preference_tuner] Processing question 1/5: What is Scott's background in machine learning?...
[preference_tuner] Pair 1: chosen=1292 chars, rejected=99 chars
[preference_tuner] Processing question 2/5: What programming languages and frameworks does Scott use?...
[preference_tuner] Pair 2: chosen=1270 chars, rejected=129 chars
[preference_tuner] Processing question 3/5: Describe Scott's most significant technical project....
[preference_tuner] Pair 3: chosen=1319 chars, rejected=114 chars
[preference_tuner] Processing question 4/5: What makes Scott a strong candidate for an ML Engineering ro...
[preference_tuner] Pair 4: chosen=1356 chars, rejected=125 chars
[preference_tuner] Processing question 5/5: How does Scott approach learning new technologies?...
[preference_tuner] Pair 5: c

In [4]:
# Inspect each preference pair
for i, pair in enumerate(dpo_pairs):
    print(f"\n{'='*60}")
    print(f"Pair {i+1}: {pair['prompt']}")
    print(f"\n[CHOSEN] ({len(pair['chosen'])} chars):")
    print(pair['chosen'][:300] + ('...' if len(pair['chosen']) > 300 else ''))
    print(f"\n[REJECTED] ({len(pair['rejected'])} chars):")
    print(pair['rejected'][:200] + ('...' if len(pair['rejected']) > 200 else ''))


Pair 1: What is Scott's background in machine learning?

[CHOSEN] (1292 chars):
I appreciate you asking, but I need to be straightforward with you:

**I don't have enough information to answer this question accurately.**

The name "Scott" is extremely common, and without more context, I cannot determine which specific person you're referring to. There are many people named Scot...

[REJECTED] (99 chars):
I don't have enough context about which Scott you're referring to — could you provide more details?

Pair 2: What programming languages and frameworks does Scott use?

[CHOSEN] (1270 chars):
I appreciate you asking, but I need to be straightforward with you:

**I don't have enough context to answer this question accurately.**

The name "Scott" is extremely common, and without more context, I cannot determine which Scott you're referring to. This could be:

- **Scott Hanselman** (Microso...

[REJECTED] (129 chars):
I don't have any information about which specific Scott you're referrin

In [5]:
# Convert to HuggingFace Dataset for DPOTrainer
from datasets import Dataset

dpo_dataset = Dataset.from_list(dpo_pairs)
print(f"DPO dataset: {len(dpo_dataset)} examples")
print(f"Columns: {dpo_dataset.column_names}")

# DPOTrainer expects: prompt, chosen, rejected
assert 'prompt' in dpo_dataset.column_names
assert 'chosen' in dpo_dataset.column_names
assert 'rejected' in dpo_dataset.column_names
print("Dataset schema validated.")

DPO dataset: 5 examples
Columns: ['prompt', 'chosen', 'rejected']
Dataset schema validated.


---
## Part 3: DPO Training

We train on top of the SFT adapter from Notebook 05. This is the standard "SFT first, then DPO" pipeline:

1. SFT: teach the model *what* to say (domain knowledge, format)
2. DPO: teach the model *which style* is preferred (thoroughness, specificity)

With `max_steps=10` this is another smoke test. Real DPO runs on thousands of pairs for 1-3 epochs.

In [6]:
from src.preference_tuner import PrefRunner

pref_runner = PrefRunner(method='dpo', base_model_path='../outputs/sft_adapter')

pref_results = pref_runner.train(
    dataset=dpo_dataset,
    output_dir='../outputs/dpo_adapter',
    max_steps=10,
    use_4bit=True,
)

print("\nDPO training results:")
print(pref_results)

[preference_tuner] Initialized PrefRunner method=dpo model=../outputs/sft_adapter
[preference_tuner] Starting DPO training, output=../outputs/dpo_adapter
[preference_tuner] DPO: loading TRL DPOTrainer...
[preference_tuner] WARNING: '../outputs/sft_adapter' is an MLX-format LoRA adapter (produced by mlx-lm.lora). HuggingFace transformers cannot load MLX adapters directly. Starting DPO from base model 'Qwen/Qwen2.5-0.5B-Instruct' instead. This is fine for the homework's pedagogical goal.
[preference_tuner] Mac detected: disabling 4-bit quantization
[preference_tuner] Loading base model: Qwen/Qwen2.5-0.5B-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[preference_tuner] Model and tokenizer loaded
[preference_tuner] Starting DPO training...


Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/scottlai/Documents/inferenceAI/Homework5-Submission/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory wo

Step,Training Loss
5,0.665455
10,0.644745


[preference_tuner] DPO complete: loss=0.6551

DPO training results:
{'method': 'dpo', 'train_loss': 0.6551000118255615, 'steps': 10, 'adapter_path': '../outputs/dpo_adapter'}


---
## Part 4: GRPO with Verifiable Rewards

GRPO is the technique that powered DeepSeek-R1 and made "reasoning models" practical at scale. The key insight:

**You don't need a reward model if you have a reward function.**

For math problems, you can check if the model's answer matches the correct answer. For code, you can run the code. For resume Q&A, you could check if key facts appear in the response.

GRPO workflow:
1. Sample `G` completions per prompt (the "group")
2. Score each with the reward function → R₁, R₂, ..., R_G
3. Compute *relative* advantage: Aᵢ = (Rᵢ - mean(R)) / std(R)
4. Update policy to increase probability of high-advantage completions

The `make_gsm8k_reward_fn` function implements the GSM8K-style reward: a completion scores **1.0** if it contains `#### <number>` (the standard GSM8K answer format), and **0.0** otherwise.

In [7]:
from src.preference_tuner import make_gsm8k_reward_fn

# Inspect the reward function
reward_fn = make_gsm8k_reward_fn()

# Example inputs and outputs
test_prompts = [
    "What is 15 + 27?",
    "If a train travels 60mph for 2 hours, how far does it go?",
    "What is 100 divided by 4?",
]
test_completions = [
    # Good: includes #### format
    "Let me think step by step. 15 + 27 = 42. #### 42",
    # Bad: correct answer but wrong format
    "The train travels 120 miles.",
    # Good: includes #### format
    "100 / 4 = 25. #### 25",
]

print("GRPO Reward Function: make_gsm8k_reward_fn")
print("Signature: reward_fn(prompts: list[str], completions: list[str]) -> list[float]")
print("Logic: return 1.0 if completion contains '#### <number>', else 0.0")
print()

rewards = reward_fn(test_prompts, test_completions)

print("\nExample results:")
print(f"{'Completion':<55} {'Reward':>8}")
print("-" * 65)
for comp, reward in zip(test_completions, rewards):
    short = comp[:52] + '...' if len(comp) > 52 else comp
    print(f"{short:<55} {reward:>8.1f}")

print()
print("For resume Q&A, you could build a reward function that checks:")
print("  - Does the response mention the candidate's name?")
print("  - Does it include specific technologies (Python, PyTorch, etc.)?")
print("  - Is it longer than 50 words (proxy for thoroughness)?")

GRPO Reward Function: make_gsm8k_reward_fn
Signature: reward_fn(prompts: list[str], completions: list[str]) -> list[float]
Logic: return 1.0 if completion contains '#### <number>', else 0.0

[preference_tuner] GRPO reward batch: 2/3 correct format

Example results:
Completion                                                Reward
-----------------------------------------------------------------
Let me think step by step. 15 + 27 = 42. #### 42             1.0
The train travels 120 miles.                                 0.0
100 / 4 = 25. #### 25                                        1.0

For resume Q&A, you could build a reward function that checks:
  - Does the response mention the candidate's name?
  - Does it include specific technologies (Python, PyTorch, etc.)?
  - Is it longer than 50 words (proxy for thoroughness)?


In [8]:
# Bonus: a simple resume Q&A reward function to illustrate the concept
def make_resume_qa_reward_fn(required_keywords: list[str] = None, min_words: int = 30):
    """Return a reward function for resume Q&A quality.
    
    Scores a completion 1.0 if it:
    - Contains at least one required keyword (if provided)
    - Has at least min_words words
    Otherwise 0.0.
    """
    keywords = [k.lower() for k in (required_keywords or [])]

    def resume_reward(prompts: list[str], completions: list[str], **kwargs) -> list[float]:
        rewards = []
        for comp in completions:
            word_count = len(comp.split())
            has_min_length = word_count >= min_words
            has_keyword = (
                any(kw in comp.lower() for kw in keywords)
                if keywords else True
            )
            score = 1.0 if (has_min_length and has_keyword) else 0.0
            rewards.append(score)
        print(f"[resume_reward] batch size={len(completions)}, avg_score={sum(rewards)/len(rewards):.2f}")
        return rewards

    return resume_reward

# Demonstrate
resume_reward = make_resume_qa_reward_fn(
    required_keywords=['python', 'machine learning', 'ml'],
    min_words=30
)
demo_completions = [
    "Scott has strong Python and machine learning skills, with experience in NLP and deep learning using PyTorch.",
    "He knows ML.",
    "Scott is proficient in Python with over 3 years of hands-on machine learning experience across NLP and computer vision tasks.",
]
demo_rewards = resume_reward(['Q'] * 3, demo_completions)
for comp, r in zip(demo_completions, demo_rewards):
    print(f"  Score {r:.0f}: {comp[:70]}..." if len(comp) > 70 else f"  Score {r:.0f}: {comp}")

[resume_reward] batch size=3, avg_score=0.00
  Score 0: Scott has strong Python and machine learning skills, with experience i...
  Score 0: He knows ML.
  Score 0: Scott is proficient in Python with over 3 years of hands-on machine le...


### TODO 1: Manual Preference Pairs

Generate 3 additional DPO preference pairs **manually** (hardcode them — no API call needed).

Think carefully about what makes a "rejected" answer bad for resume Q&A. It shouldn't just be shorter — it should fail in specific ways.

Common failure modes for rejected answers:
- Too vague: "He has experience" (experience with what?)
- Contradicts facts: wrong tech stack, wrong time period
- Wrong tone: overly casual, overly corporate/buzzword-heavy
- Incomplete: stops mid-answer without addressing the question

In your answer, also explain which failure mode you chose and why.

In [9]:
# TODO 1: Hardcode 3 additional DPO preference pairs
manual_pairs = [
    {
        "prompt": "What technologies does Scott use for building LLM applications?",
        "chosen": "YOUR GOOD ANSWER HERE (specific, mentions real tools like Claude API, FAISS, TRL, MLX-LM)",
        "rejected": "YOUR BAD ANSWER HERE (vague, e.g., 'He uses various AI tools')",
    },
    {
        "prompt": "What is Scott's experience with fine-tuning language models?",
        "chosen": "YOUR GOOD ANSWER HERE",
        "rejected": "YOUR BAD ANSWER HERE",
    },
    {
        "prompt": "What are Scott's goals for the next year in his career?",
        "chosen": "YOUR GOOD ANSWER HERE",
        "rejected": "YOUR BAD ANSWER HERE",
    },
]

# Failure mode explanation
failure_mode_explanation = """
The rejected answers fail because:
[YOUR EXPLANATION: which failure mode(s) did you use and why are they harmful?]
"""

print(f"Manual pairs: {len(manual_pairs)}")
print(failure_mode_explanation)

Manual pairs: 3

The rejected answers fail because:
[YOUR EXPLANATION: which failure mode(s) did you use and why are they harmful?]



In [10]:
# TODO 1 reflection -- edit your answer below, then run this cell.
# (The manual_pairs list above is your data; this cell captures your written reasoning.)
todo1_reflection = """[Replace with your written reflection on TODO 1: manual preference pairs]

Hint: Briefly describe the 3 manual pairs you authored above. For each one, name the
specific failure mode you used in the rejected answer (e.g., too vague, contradicts facts,
wrong tone, incomplete) and explain in 1-2 sentences why that failure mode is harmful for
a resume Q&A assistant in production. Avoid the trap of just making rejected answers
shorter -- the rejection should fail in a content-meaningful way.
"""
print(todo1_reflection)


[Replace with your written reflection on TODO 1: manual preference pairs]

Hint: Briefly describe the 3 manual pairs you authored above. For each one, name the
specific failure mode you used in the rejected answer (e.g., too vague, contradicts facts,
wrong tone, incomplete) and explain in 1-2 sentences why that failure mode is harmful for
a resume Q&A assistant in production. Avoid the trap of just making rejected answers
shorter -- the rejection should fail in a content-meaningful way.



### TODO 2: KTO vs DPO

Give a **concrete scenario** where you would choose KTO over DPO for your resume Q&A assistant.

Constraints:
- The scenario must involve a real data collection situation (not just "KTO needs less data")
- Explain why you only have binary labels (thumbs up/down) rather than explicit chosen/rejected pairs
- What would be the practical steps to collect and format this KTO data?

In [11]:
# TODO 2: KTO vs DPO scenario
todo2_response = """
Scenario where I would use KTO over DPO:

[Describe the scenario — e.g., you deployed a Slack bot answering resume questions,
 users can only react with thumbs-up or thumbs-down emoji]

Why only binary labels:
[Explain the UX/interface constraint]

How to collect and format KTO data:
- Step 1: [e.g., log each bot response with the prompt]
- Step 2: [e.g., capture the emoji reaction as label=True/False]
- Step 3: [e.g., format as {"prompt": ..., "completion": ..., "label": True/False}]
"""
print(todo2_response)


Scenario where I would use KTO over DPO:

[Describe the scenario — e.g., you deployed a Slack bot answering resume questions,
 users can only react with thumbs-up or thumbs-down emoji]

Why only binary labels:
[Explain the UX/interface constraint]

How to collect and format KTO data:
- Step 1: [e.g., log each bot response with the prompt]
- Step 2: [e.g., capture the emoji reaction as label=True/False]
- Step 3: [e.g., format as {"prompt": ..., "completion": ..., "label": True/False}]



In [12]:
# TODO 2 reflection -- edit your answer below, then run this cell.
todo2_reflection = """[Replace with your answer to TODO 2: a concrete scenario where KTO beats DPO]

Hint: Describe a real data-collection situation (e.g., a Slack bot where users only react
with thumbs-up / thumbs-down emoji), explain why the UX gives you binary labels rather than
explicit chosen/rejected pairs, and list 3 concrete steps to collect and format the data
into KTO's {"prompt", "completion", "label": True/False} schema. Avoid the generic answer
"KTO needs less data" -- ground it in a specific product surface.
"""
print(todo2_reflection)


[Replace with your answer to TODO 2: a concrete scenario where KTO beats DPO]

Hint: Describe a real data-collection situation (e.g., a Slack bot where users only react
with thumbs-up / thumbs-down emoji), explain why the UX gives you binary labels rather than
explicit chosen/rejected pairs, and list 3 concrete steps to collect and format the data
into KTO's {"prompt", "completion", "label": True/False} schema. Avoid the generic answer
"KTO needs less data" -- ground it in a specific product surface.



---
## Summary

In this notebook you:
- Understood why preference tuning is needed beyond SFT (alignment tax)
- Compared DPO, KTO, and GRPO across data requirements and use cases
- Generated 5 DPO preference pairs using `PrefRunner.build_dpo_dataset_from_llm`
- Ran a DPO smoke test on top of the SFT adapter
- Inspected the GRPO reward function and designed a custom resume Q&A reward

**Key takeaways:**
- **DPO** = best quality per compute dollar when you can generate (chosen, rejected) pairs
- **KTO** = best when you only have thumbs-up/thumbs-down from real users
- **GRPO** = best for tasks with verifiable correctness (math, code, structured output)
- The full stack is: **Pretrain → Mid-train → SFT → Preference Tuning → Eval → Serve**

Week 5 complete — you've run the entire post-training pipeline on a real model!

In [13]:
# Save preference tuning results + auto-capture TODO reflections
import json

output_data = {
    'dpo_training': pref_results if 'pref_results' in dir() else {},
    'dpo_dataset_size': len(dpo_pairs) if 'dpo_pairs' in dir() else 0,
    'manual_pairs_count': len(manual_pairs) if 'manual_pairs' in dir() else 0,
    'grpo_reward_fn': 'make_gsm8k_reward_fn (pattern: ####\\s*-?\\d+)',
    'resume_reward_fn': 'make_resume_qa_reward_fn (keywords + min_words check)',
}

os.makedirs('../outputs', exist_ok=True)
with open('../outputs/preference_tuning_results.json', 'w') as f:
    json.dump(output_data, f, indent=2)
print("Saved outputs/preference_tuning_results.json")

# Build reflection from student TODO answers (auto-captured)
section_text = (
    "### TODO 1: Manual Preference Pairs\n" + (todo1_reflection if 'todo1_reflection' in dir() else "[not completed]") + "\n\n" +
    "### TODO 2: KTO vs DPO Scenario\n" + (todo2_reflection if 'todo2_reflection' in dir() else "[not completed]")
)
append_to_reflection(
    notebook="06",
    section_title="Preference Tuning: DPO, KTO & GRPO",
    reflection_content=section_text,
    output_dir="../outputs",
)
print("Reflection auto-saved to outputs/homework_reflection.md")
tracker.report()


Saved outputs/preference_tuning_results.json
Reflection auto-saved to outputs/homework_reflection.md
API COST REPORT
Total API calls:     0
Total input tokens:  0
Total output tokens: 0
Total cost:          $0.0000

